# 🌱 Create Model‑Organism & Human Gene Nodes with Orthology Links

This notebook processes the GeneLab datasets to build Neo4j node and relationship files for model‑organism genes (MGene), human ortholog genes, and their orthology links. It uses the `genelab_utils` and `ortholog_mapper` packages to extract gene IDs, map to human orthologs, and write CSVs ready for SPOKE ingestion.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [15]:
import pandas as pd
import genelab_utils as gl
import ortholog_mapper

In [16]:
pd.set_option('display.max_rows', None)  # Shows all rows
pd.set_option('display.max_colwidth', None)  # Shows full content of each cell

## Setup Environment Variables
Edit `../.env` to configure the environment.   

In [17]:
# Node and relationship directory paths
node_dir, rel_dir = gl.setup_environment()

Environment setup for KG version: v0.3.1


In [19]:
MANIFEST_PATH = "../data/manifest.csv" # file with dataset info

## Get Info about available Datasets

In [20]:
manifest = pd.read_csv(MANIFEST_PATH)
manifest.head()

,identifier,technology,measurement,assay_name,taxonomy,material,host_organism,host_strain,filename,url,differential_analysis_method,organism,host_taxonomy
0,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_Affymetrix,7227,Whole Organism,NaN,NaN,GLDS-1_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-1/download?source=datamanager&file=GLDS-1_array_differential_expression.csv,DESeq2,Drosophila melanogaster,NaN
1,OSD-3,DNA microarray,transcription profiling,OSD-3_transcription-profiling_dna-microarray_affymetrix,7227,Whole Organism,NaN,NaN,GLDS-3_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-3/download?source=datamanager&file=GLDS-3_array_differential_expression_GLmicroarray.csv,DESeq2,Drosophila melanogaster,NaN
2,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_affymetrix,10090,thymus,NaN,NaN,GLDS-4_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-4/download?source=datamanager&file=GLDS-4_array_differential_expression.csv,DESeq2,Mus musculus,NaN
3,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_affymetrix,10116,Cells,NaN,NaN,GLDS-6_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-6/download?source=datamanager&file=GLDS-6_array_differential_expression_GLmicroarray.csv,DESeq2,Rattus norvegicus,NaN
4,OSD-13,DNA microarray,transcription profiling,OSD-13_transcription-profiling_dna-microarray_Affymetrix,9606,Primary T Cells,NaN,NaN,GLDS-13_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-13/download?source=datamanager&file=GLDS-13_array_differential_expression_GLmicroarray.csv,DESeq2,Homo sapiens,NaN


## Create MGene (Model Organism Gene) Nodes

In [21]:
# Parse all data files for gene ids (ENTREZID) and create a unique list of all genes.
mgenes = gl.extract_gene_info(manifest)

In [22]:
mgene_nodes = gl.save_dataframe_to_kg(mgenes, 'MGene', node_dir)
print(f"Number of MGene nodes: {mgene_nodes.shape[0]}")
mgene_nodes.head()

Number of MGene nodes: 124549


,identifier,symbol,name,organism,taxonomy
0,42037,abd-A,abdominal A,Drosophila melanogaster,7227
1,38545,nAChRbeta1,nicotinic Acetylcholine Receptor beta1,Drosophila melanogaster,7227
2,31521,Act5C,Actin 5C,Drosophila melanogaster,7227
4,35526,Act42A,Actin 42A,Drosophila melanogaster,7227
5,33986,Gart,GART trifunctional enzyme,Drosophila melanogaster,7227


## Map Model Organism Genes to Human Orthologs
Orthologs are mapped using the taxonomy id and ENTREZ gene identifier. Suggestions for suitable ortholog databases are listed below.

In [23]:
# List of taxonomies in the dataset with statistically significant data that have ENTREZ IDs
print(f"List of taxonomy ids in the datasets: {mgenes['taxonomy'].unique()}")

List of taxonomy ids in the datasets: ['7227' '10090' '10116' '9606' '3702' '6239' '4932' '1423']


In [24]:
# List of supported ortholog_dbs that have a mapping to human ENTREZ IDs
suggestions = ortholog_mapper.suggest_ortholog_dbs(mgenes, "taxonomy", "identifier")
suggestions

,taxonomy,supported_dbs
0,1423,[]
1,3702,[]
2,4932,"[Panther, Ensembl, EggNOG, HomoloGene, PhylomeDB, OMA, Inparanoid]"
3,6239,"[Panther, Ensembl, EggNOG, HomoloGene, PhylomeDB, Treefam, OMA, OrthoDB, Inparanoid]"
4,7227,"[Panther, Ensembl, EggNOG, HomoloGene, PhylomeDB, Treefam, OMA, OrthoDB, Inparanoid]"
5,10090,"[Panther, HGNC, Ensembl, EggNOG, HomoloGene, PhylomeDB, Treefam, JAX, OMA, OrthoDB, NCBI, Inparanoid]"
6,10116,"[Panther, Ensembl, EggNOG, HomoloGene, PhylomeDB, Treefam, JAX, OMA, OrthoDB, NCBI, Inparanoid]"


In [25]:
mapped_genes = ortholog_mapper.map_orthologs(mgenes, "taxonomy", "identifier", "human_entrez_id", ortholog_dbs=["JAX", "Ensembl"])
# Remove any genes that cannot be mapped to human ENTREZ ids
mapped_genes = mapped_genes[mapped_genes["human_entrez_id"] != ""]

## Create Human Ortholog Gene Nodes
The mapped genes include non-protein coding genes

In [26]:
human_genes = mapped_genes[['human_entrez_id']].copy()
human_genes.rename(columns={'human_entrez_id': 'identifier'}, inplace=True)

In [27]:
human_gene_nodes = gl.save_dataframe_to_kg(human_genes, 'Gene', node_dir)
print(f"Number of Gene nodes: {human_gene_nodes.shape[0]}")
human_gene_nodes.head()

Number of Gene nodes: 30170


,identifier
2,60
3,71
6,2618
7,226
8,229


## Create MGene-IS_ORTHOLOG_MGiG-Gene Relationships

In [28]:
model_to_human_genes = mapped_genes[["identifier", "human_entrez_id"]].copy()
model_to_human_genes.rename(columns={"identifier": "from", "human_entrez_id": "to"}, inplace=True)

In [29]:
model_to_human_gene_rels = gl.save_dataframe_to_kg(model_to_human_genes, 'MGene-IS_ORTHOLOG_MGiG-Gene', rel_dir)
print(f"Number of MGene-IS_ORTHOLOG_MGiG-Gene relationships: {model_to_human_gene_rels.shape[0]}")
model_to_human_gene_rels.head()

Number of MGene-IS_ORTHOLOG_MGiG-Gene relationships: 102699


,from,to
2,31521,60
3,31521,71
4,35526,60
5,35526,71
6,33986,2618
